In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import joblib
import os

# caricamento dati
data = pd.read_csv("/content/drive/MyDrive/prototipo2/tickets_sintetici.csv")
data["title"] = data["title"].fillna("")
data["title"] = data["title"].str.lower()
data["body"] = data["body"].fillna("")
data["body"] = data["body"].str.lower()
data["text"] = data["title"] + " " + data["body"]

In [3]:
priority_map = {
    "bassa": 0,
    "media": 1,
    "alta": 2
}

y = data["priority"].map(priority_map).values

In [4]:
vectorizer = joblib.load(
    "/content/drive/MyDrive/prototipo2/models/tfidf_vectorizer.joblib"
)

X = vectorizer.fit_transform(data["text"]).toarray()

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train_text, X_test_text, y_train, y_test = train_test_split(
    data["text"], y, test_size=0.2, random_state=42
)
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.9
)

X_train = vectorizer.fit_transform(X_train_text).toarray()
X_test = vectorizer.transform(X_test_text).toarray()
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

In [6]:
class TicketPriorityClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model_prio = TicketPriorityClassifier(X_train.shape[1], 3)
weights = torch.tensor([1.0, 1.3, 1.0])
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(
    model_prio.parameters(),
    lr=0.001,
    weight_decay=1e-4)

In [7]:
for epoch in range(31):
    optimizer.zero_grad()
    outputs = model_prio(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 1.0991
Epoch 10, Loss: 1.0452
Epoch 20, Loss: 0.8538
Epoch 30, Loss: 0.4858


In [8]:
with torch.no_grad():
    outputs = model_prio(X_test)
    preds = torch.argmax(outputs, dim=1)

from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, preds))


Accuracy: 0.73


In [9]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, preds))

[[25  4  3]
 [ 6 29  6]
 [ 1  7 19]]


In [10]:
from sklearn.metrics import classification_report

classificazione = classification_report(y_test, preds)
print(classificazione)

              precision    recall  f1-score   support

           0       0.78      0.78      0.78        32
           1       0.72      0.71      0.72        41
           2       0.68      0.70      0.69        27

    accuracy                           0.73       100
   macro avg       0.73      0.73      0.73       100
weighted avg       0.73      0.73      0.73       100



In [ ]:
MODEL_DIR = "/content/drive/MyDrive/prototipo2/models"
os.makedirs(MODEL_DIR, exist_ok=True)

torch.save(
    model_prio.state_dict(),
    MODEL_DIR + "/priority_model.pt"
)
joblib.dump(
    vectorizer,
    MODEL_DIR + "/tfidf_prio_vectorizer.joblib"
)

print("✅ priority_model.pt salvato")


✅ priority_model.pt salvato


Questa cella monta Google Drive nel file system di Colab, consentendo l'accesso ai file salvati sul tuo Drive, come il dataset e i modelli.

Questa cella carica il dataset `tickets_sintetici.csv` in un DataFrame pandas. Esegue poi il preprocessing iniziale dei campi 'title' e 'body': riempie i valori mancanti con stringhe vuote, li converte in minuscolo e li concatena in una nuova colonna 'text' che verrà utilizzata per la vettorizzazione.

Qui viene definita una mappa per convertire le etichette di priorità testuali (bassa, media, alta) in valori numerici (0, 1, 2), necessari per l'addestramento del modello. Successivamente, la colonna 'priority' del DataFrame viene mappata utilizzando questa definizione e i valori risultanti vengono assegnati alla variabile `y`.

Questa cella divide il dataset in set di training e test per i dati di testo (`X_train_text`, `X_test_text`) e per le etichette (`y_train`, `y_test`). Successivamente, inizializza un `TfidfVectorizer` per convertire il testo in vettori numerici. Addestra il `vectorizer` sul set di training e lo trasforma, quindi trasforma il set di test. Infine, converte i vettori TF-IDF risultanti in tensori PyTorch di tipo `float32`.

Questa cella definisce l'architettura della rete neurale `TicketPriorityClassifier` utilizzando PyTorch. Il modello è una rete feed-forward con due strati nascosti, funzioni di attivazione ReLU e dropout per prevenire l'overfitting. Inizializza il modello, definisce i pesi per la funzione di costo `CrossEntropyLoss` (per gestire eventuali squilibri tra le classi di priorità) e configura l'ottimizzatore Adam con un learning rate e un weight decay specifici.

Questa cella contiene il loop di addestramento del modello. Per 31 epoche, calcola le previsioni del modello sul set di training, la perdita utilizzando la `CrossEntropyLoss`, esegue la backpropagation per calcolare i gradienti e aggiorna i pesi del modello tramite l'ottimizzatore Adam. Ogni 10 epoche, stampa la perdita corrente per monitorare l'andamento dell'addestramento.

Dopo l'addestramento, questa cella valuta le prestazioni del modello sul set di test. Utilizza il modello per fare previsioni (`preds`) sul `X_test` e calcola l'accuratezza confrontando le previsioni con le etichette reali (`y_test`) utilizzando `accuracy_score` di scikit-learn.

Questa cella calcola e stampa la matrice di confusione, una metrica dettagliata che mostra il numero di classificazioni corrette e errate per ogni classe di priorità. È utile per comprendere dove il modello sta commettendo errori.

Infine, questa cella salva lo stato (i pesi addestrati) del modello di PyTorch in un file chiamato `priority_model.pt` nella cartella `/content/drive/MyDrive/prototipo2/models`. Questo permette di riutilizzare il modello addestrato senza doverlo addestrare nuovamente in futuro.